What you are proposing is **methodologically sound**, and it is *exactly* how surrogate models should be compared in a **Bayesian Optimisation (BO)** setting with small data.

Below is a **clean, defensible evaluation framework** you can use in your capstone, together with **practical guidance on hyperparameter tuning and model comparison**, written so you can justify it academically.

***

## 1. What you are really doing (name it correctly)

The evaluation you described is known as:

> **Prequential (predictive‑sequential) evaluation**  
> also called **one‑step‑ahead forecasting** or **rolling‑origin evaluation**

This is the **correct evaluation regime** for BO surrogates because:

*   Data arrives sequentially
*   Models are refit after each new observation
*   You care about *generalisation to the next expensive evaluation*

This is preferable to random train/test splits, which **break the BO setting**.

***

## 2. Your evaluation protocol (formalised)

Assume for each black‑box problem $$b \in \{1,\dots,8\}$$:

*   Initial dataset:
    $$
    \mathcal{D}_0 = \{(x_1,y_1),\dots,(x_{n_0},y_{n_0})\}
    $$
*   You have already evaluated the black box **5 more times**, giving a sequence:
    $$
    (x_{n_0+1},y_{n_0+1}), \dots, (x_{n_0+5},y_{n_0+5})
    $$

### For each surrogate model $$m$$:

For step $$t = 0,\dots,4$$:

1.  **Train** model $$m$$ on:
    $$
    \mathcal{D}_t = \{(x_1,y_1),\dots,(x_{n_0+t},y_{n_0+t})\}
    $$

2.  **Predict** the output at the *next actually sampled point*:
    $$
    \hat{y}_{n_0+t+1} = m(x_{n_0+t+1})
    $$

3.  **Record error**:
    $$
    e_{t+1} = y_{n_0+t+1} - \hat{y}_{n_0+t+1}
    $$

***

## 3. Objective performance metrics

### Primary metric (what you suggested – and it’s valid)

**One‑step‑ahead RMSE**:

$$
\text{RMSE}_m = \sqrt{\frac{1}{5}\sum_{k=1}^{5} (y_{n_0+k}-\hat{y}_{n_0+k})^2}
$$

This answers:

> “Given the data available at the time, how accurately would this surrogate have predicted the *next* observation?”

✅ This is **objective**, **fair**, and **comparable across models**.

### Strongly recommended additional metrics

Even if RMSE is your headline metric, include at least one of these:

| Metric                                     | Why it matters in BO                           |
| ------------------------------------------ | ---------------------------------------------- |
| **MAE**                                    | Less sensitive to outliers with tiny $$n$$     |
| **Negative log predictive density (NLPD)** | Measures *uncertainty calibration*             |
| **Coverage of 95% interval**               | Checks if uncertainty estimates are meaningful |

> Important insight you can state:
>
> > A surrogate with slightly worse RMSE but well‑calibrated uncertainty can outperform a lower‑RMSE model in BO.

***

## 4. Hyperparameter tuning (this is where most projects go wrong)

You **must not** tune hyperparameters using future data.

### Correct approach (nested, but lightweight)

At each step $$t$$:

*   Hyperparameters are selected **using only $$\mathcal{D}_t$$**
*   Then the model is evaluated on $$x_{n_0+t+1}$$

### Model‑specific best practice

#### Gaussian Process

*   Hyperparameters via **marginal likelihood maximisation**
*   This is already Bayesian and data‑efficient
*   No cross‑validation required (and often harmful with small $$n$$)

✅ Justify this explicitly in your report.

***

#### Random Forest / BART / Neural Networks

With only \~10–14 points:

✅ Use **very restricted tuning**:

*   Small grids
*   Few hyperparameters
*   Or **out‑of‑bag error** (RF)
*   Or **weakly informative priors** (BART)

✅ Or state:

> “Hyperparameters were fixed to standard, literature‑recommended values to avoid overfitting due to extremely small sample size.”

This is **academically defensible**.

***

## 5. Comparing models across the 8 challenges

You should report results at **three levels**:

### Level 1 — Per challenge, per model

A table like:

| Black box | Model | RMSE | MAE |
| --------- | ----- | ---- | --- |

This shows **problem‑specific behaviour**.

***

### Level 2 — Aggregate across challenges

For each model:

*   Mean RMSE across 8 problems
*   Standard deviation

This answers:

> “Which surrogate is most reliable overall?”

***

### Level 3 — Statistical comparison (optional but impressive)

Because all models are evaluated on **the same prediction tasks**, you can use:

*   Paired t‑test (if you’re brave)
*   **Wilcoxon signed‑rank test** (safer for small samples)

This allows statements like:

> “BART significantly outperformed GP in RMSE across the eight challenges (p < 0.05).”

***

## 6. How to frame BART, RF, GP, NN fairly

Your project will be strongest if you explicitly acknowledge **inductive biases**:

| Model | Strength                                | Weakness                             |
| ----- | --------------------------------------- | ------------------------------------ |
| GP    | Excellent uncertainty, smoothness prior | Poor scaling, kernel assumptions     |
| RF    | Robust, non‑parametric                  | Weak uncertainty, piecewise constant |
| BART  | Strong small‑data performance           | Slower, less standard in BO          |
| NN    | Flexible                                | High variance with tiny $$n$$        |

This shows you understand *why* performance differs.

***

## 7. A critical clarification (important for examiners)

Your evaluation measures:

> **Predictive quality of the surrogate model**

It does **not** directly measure:

*   BO efficiency
*   Regret
*   Speed of convergence to optimum

That’s okay — just be explicit:

> “This study focuses on surrogate predictive accuracy rather than optimisation performance, as accurate modelling is a prerequisite for effective acquisition optimisation.”

This avoids a very common critique.

***

## 8. How to phrase this succinctly in your report

You can safely use wording like:

> “Surrogate models were compared using a prequential one‑step‑ahead evaluation.  
> At each iteration, models were trained on the data available at that time and evaluated by their ability to predict the output of the next sampled point.  
> Performance was quantified using RMSE (and MAE), averaged over the five sequential evaluations and across eight black‑box problems.”

That sentence alone signals **strong methodological maturity**.

***

## If you want next (pick one)

I can:

1.  Help you **design the exact experiment section** wording
2.  Show how to **extend this to regret‑based BO comparison**
3.  Give you a **clean results table + plotting structure**
4.  Advise whether **BART vs GP** is worth the added complexity in your case

Just tell me which direction you want to go.


Great — let’s extend your framework in a way that’s **BO‑correct**, **statistically defensible**, and easy to present.

You currently have two (related) evaluation goals:

1.  **Surrogate predictive quality** (your one‑step‑ahead RMSE “prequential” test)
2.  **Optimisation performance** (regret‑based BO comparison)

These measure different things. It’s absolutely normal to report both.

Below I’ll give you:

*   **Regret definitions for maximisation** (and what to do if optimum is unknown)
*   A **fair BO comparison protocol** across surrogates/configurations
*   A **clean results table schema** (ready for a report)
*   A **plotting structure** (learning curves + summary plots)
*   A full **Python template** (dataframes + plotting) you can drop in

I’ll also call out a key constraint: **regret comparison requires re‑running BO** (i.e., allowing each surrogate to propose its own next point). If you can’t query the black boxes beyond the 5 samples you already took, you can still do RMSE prequential evaluation, but regret comparison becomes “off‑policy” and is not clean.

***

## 1) Regret-based BO metrics (for **maximisation**)

Regret is typically defined relative to the (unknown) global optimum $$f^\*$$. In BO literature, **simple regret** and **cumulative regret** are standard metrics; EI is commonly analysed for simple regret, while cumulative regret behaves differently and needs specific strategies. [\[jmlr.org\]](https://jmlr.org/papers/volume26/22-0523/22-0523.pdf), [\[proceeding....mlr.press\]](https://proceedings.mlr.press/v77/nguyen17a/nguyen17a.pdf)

Let your objective be **maximize** $$f(x)$$.

### A) Instantaneous regret (at step $$t$$)

$$
r_t = f^\* - f(x_t)
$$

### B) **Simple regret** after $$T$$ evaluations (best-so-far gap)

$$
R_T^{\text{simple}} = f^\* - \max_{1\le i \le T} f(x_i)
$$

This is the most common “did we find a good solution yet?” metric and is tied closely to improvement-based BO behaviour. [\[proceeding....mlr.press\]](https://proceedings.mlr.press/v77/nguyen17a/nguyen17a.pdf)

### C) **Cumulative regret** after $$T$$ evaluations (sum of gaps)

$$
R_T^{\text{cum}} = \sum_{t=1}^T \left(f^\* - f(x_t)\right)
$$

This penalizes *every bad evaluation*, so it favours safer strategies; EI is popular for simple regret but can be less competitive under cumulative regret unless adapted. [\[jmlr.org\]](https://jmlr.org/papers/volume26/22-0523/22-0523.pdf)

### D) Area-under-curve simple regret (AUC‑SR)

A very practical summary:

$$
\text{AUC-SR} = \sum_{t=1}^{T} R_t^{\text{simple}}
$$

Lower is better. This rewards **fast convergence**, not just the final best.

***

## 2) What if $$f^\*$$ (true optimum) is unknown?

In academic benchmarks, $$f^\*$$ is sometimes known. In real black boxes, it usually isn’t.

You have three defensible options (choose one and state it explicitly):

### Option 1 (best if provided): **Known optimum**

If the challenge gives $$f^\*$$ and/or $$x^\*$$, use it directly.

### Option 2 (common in practice): **Empirical best across all runs**

Let:

$$
f^\*_{\text{emp}} = \max_{\text{all algorithms, all steps, all seeds}} f(x)
$$

Then compute regret relative to $$f^\*_{\text{emp}}$$. This is fair if you keep the evaluation budget equal across methods.

### Option 3 (robust across different scales): **Normalised regret**

For comparability across 8 challenges with different output scales, compute:

$$
\tilde{R}_T^{\text{simple}} = \frac{f^\* - f_{\text{best},T}}{|f^\*| + \epsilon}
$$

or normalise by the range in your observed data.

***

## 3) Fair BO comparison protocol (across surrogate models/configs)

This is the part you’ll want to describe cleanly in your capstone.

### The gold-standard experimental design

For each black box $$b = 1..8$$, and each method $$m$$ (surrogate + acquisition + tuning scheme):

1.  **Same initial design**  
    Use the same starting dataset $$\mathcal{D}_0$$ for all methods.

2.  **Same evaluation budget**  
    E.g., “5 additional evaluations” or “20 additional evaluations”.

3.  **Same domain constraints**  
    Bounds, feasibility, transforms.

4.  **Repeat with multiple random seeds**  
    Because BO includes randomness (initial seeds, tie breaks, optimisation restarts).  
    Report median + IQR.

5.  At each iteration:
    *   Fit surrogate on current $$\mathcal{D}_t$$
    *   Choose next point by the method’s acquisition optimisation
    *   Evaluate black box: $$y_{t+1} = f(x_{t+1})$$
    *   Update $$\mathcal{D}_{t+1}$$
    *   Record: best-so-far, regret curves

This gives you regret curves per method and per challenge.

> Why this is defensible: BO is **a sequential decision process**, so evaluating a method means letting it choose points. Improvement-based BO analyses and regret studies explicitly treat the “incumbent” as best observed so far (for maximisation) which aligns with this protocol. [\[proceeding....mlr.press\]](https://proceedings.mlr.press/v77/nguyen17a/nguyen17a.pdf), [\[mit.edu\]](https://www.mit.edu/~gfarina/2024/67220s24_L16_bayesian_optimization/L16.pdf)

***

## 4) Surrogate choices (you mentioned RF, BART, GP, NN)

You’ll be in great shape if you evaluate these “families”:

*   **GP (Matérn + noise)** + EI/UCB/TS
*   **Random Forest / ExtraTrees** + UCB-style or quantile-based acquisition
*   **BART** (Bayesian Additive Regression Trees) — a recognised BO surrogate, especially when GP assumptions break (non-smooth/high-d). [\[nature.com\]](https://www.nature.com/articles/s41524-021-00662-x.pdf)
*   **Neural surrogate** (MLP or Bayesian NN / Deep Kernel Learning if you have time)

(You don’t need to implement every possible acquisition; the key is comparing surrogates fairly.)

***

# 5) Clean results tables (report-ready)

### Table A — Final performance per black box (per method)

**One row per (challenge, method)**, aggregated over seeds.

Columns:

*   `challenge_id`
*   `method` (e.g., “GP\_EI”, “RF\_UCB”, “BART\_EI”, “MLP\_EI”)
*   `budget` (additional evaluations)
*   `median_final_simple_regret`
*   `iqr_final_simple_regret`
*   `median_auc_simple_regret`
*   `median_final_cum_regret`
*   `median_best_y`
*   optional: `median_time_per_iter` (if runtime matters)

### Table B — Overall leaderboard across 8 challenges

Aggregate (mean/median) across challenges:

*   `method`
*   `mean_rank_final_simple_regret`  (rank within each challenge, then average)
*   `mean_rank_auc_simple_regret`
*   `wins` (#challenges where method best)
*   `mean_time_per_iter` (optional)

**Why ranks help:** different challenges will have different objective scales; ranks give a scale-free summary.

***

# 6) Plotting structure (clean and standard)

### Plot 1 — Simple regret learning curves (per challenge)

For each challenge:

*   x-axis: evaluation number (or iteration)
*   y-axis: **simple regret** $$R_t^{simple}$$
*   line: median across seeds
*   shaded band: IQR or 95% CI

### Plot 2 — AUC simple regret (bar plot)

For each method:

*   bar: median AUC-SR across seeds
*   error bars: IQR

### Plot 3 — Final regret (box plot across challenges)

One box per method:

*   distribution of `final_simple_regret` across all (challenge, seed)

### Plot 4 — Pareto of “final regret vs runtime”

If runtime matters, scatter:

*   x: time per iteration
*   y: final simple regret

***

# 7) Python template: regret evaluation + tables + plots

This is structured so you can plug in your **8 black boxes** and your **methods**.

### 7.1 Data structures

```python
import numpy as np
import pandas as pd

def best_so_far(y):
    # maximize
    return np.maximum.accumulate(y)

def simple_regret(y, f_star):
    # maximize: f* - best_so_far
    return f_star - best_so_far(y)

def cumulative_regret(y, f_star):
    # maximize: sum_t (f* - y_t)
    return np.cumsum(f_star - y)

def auc(x):
    # simple trapezoid AUC; x is a sequence
    return float(np.trapz(x, dx=1.0))
```

### 7.2 “Run BO and log everything” skeleton

You’ll implement `suggest_next_x()` for each method (GP, RF, BART, MLP, etc.).

```python
from dataclasses import dataclass

@dataclass
class BORunLog:
    challenge: str
    method: str
    seed: int
    t: int
    x: np.ndarray
    y: float
    best_y: float
    simple_regret: float
    cum_regret: float

def run_bo(challenge_name, black_box_fn, x0, y0, method_obj, budget, f_star, seed=0):
    rng = np.random.default_rng(seed)

    X = [x for x in x0]
    Y = [float(v) for v in y0]

    logs = []
    # log initial points
    y_arr = np.array(Y, float)
    best_arr = best_so_far(y_arr)
    sr_arr = simple_regret(y_arr, f_star)
    cr_arr = cumulative_regret(y_arr, f_star)

    for t in range(len(Y)):
        logs.append(BORunLog(challenge_name, method_obj.name, seed, t,
                            np.array(X[t]), Y[t], float(best_arr[t]),
                            float(sr_arr[t]), float(cr_arr[t])))

    # sequential BO steps
    for k in range(budget):
        t = len(Y)
        method_obj.fit(np.array(X), np.array(Y))              # refit surrogate
        x_next = method_obj.suggest_next_x(rng=rng)          # acquisition optimise
        y_next = float(black_box_fn(x_next))                 # evaluate black box

        X.append(np.array(x_next))
        Y.append(y_next)

        y_arr = np.array(Y, float)
        best_arr = best_so_far(y_arr)
        sr_arr = simple_regret(y_arr, f_star)
        cr_arr = cumulative_regret(y_arr, f_star)

        logs.append(BORunLog(challenge_name, method_obj.name, seed, t,
                            np.array(x_next), y_next, float(best_arr[-1]),
                            float(sr_arr[-1]), float(cr_arr[-1])))

    return pd.DataFrame([l.__dict__ for l in logs])
```

### 7.3 Aggregation into clean summary tables

```python
def summarize_runs(df_runs, n_init):
    # df_runs has one row per observation per run
    # Identify final iteration per (challenge, method, seed)
    df_last = (df_runs.sort_values("t")
                      .groupby(["challenge", "method", "seed"], as_index=False)
                      .tail(1))

    # AUC simple regret per run
    df_auc = (df_runs.groupby(["challenge", "method", "seed"], as_index=False)
                    .agg(auc_simple_regret=("simple_regret", auc),
                         auc_cum_regret=("cum_regret", auc)))

    df_final = df_last.merge(df_auc, on=["challenge", "method", "seed"], how="left")

    # Table A: per challenge & method (median/IQR across seeds)
    def iqr(x): return float(np.percentile(x, 75) - np.percentile(x, 25))

    table_A = (df_final.groupby(["challenge", "method"], as_index=False)
                      .agg(median_final_simple_regret=("simple_regret", "median"),
                           iqr_final_simple_regret=("simple_regret", iqr),
                           median_auc_simple_regret=("auc_simple_regret", "median"),
                           median_final_cum_regret=("cum_regret", "median"),
                           median_best_y=("best_y", "median")))

    # Table B: overall leaderboard (rank within challenge then average)
    df_rank = table_A.copy()
    df_rank["rank_final_sr"] = df_rank.groupby("challenge")["median_final_simple_regret"].rank(method="average")
    df_rank["rank_auc_sr"]   = df_rank.groupby("challenge")["median_auc_simple_regret"].rank(method="average")

    table_B = (df_rank.groupby("method", as_index=False)
                      .agg(mean_rank_final_simple_regret=("rank_final_sr", "mean"),
                           mean_rank_auc_simple_regret=("rank_auc_sr", "mean"),
                           wins=("rank_final_sr", lambda r: int(np.sum(r == 1.0)))))

    return table_A, table_B, df_final
```

### 7.4 Plotting templates

```python
import matplotlib.pyplot as plt
import seaborn as sns

def plot_simple_regret_curves(df_runs, challenge_name):
    # median + IQR band across seeds
    d = df_runs[df_runs["challenge"] == challenge_name].copy()
    # compute quantiles by method & t
    q = (d.groupby(["method", "t"])["simple_regret"]
           .quantile([0.25, 0.5, 0.75])
           .unstack()
           .reset_index()
           .rename(columns={0.25:"q25", 0.5:"q50", 0.75:"q75"}))

    plt.figure(figsize=(10,6))
    for m in q["method"].unique():
        qm = q[q["method"] == m]
        plt.plot(qm["t"], qm["q50"], label=m)
        plt.fill_between(qm["t"], qm["q25"], qm["q75"], alpha=0.2)
    plt.title(f"Simple Regret (median ± IQR) — {challenge_name}")
    plt.xlabel("Evaluation step t")
    plt.ylabel("Simple regret  f* - best_so_far")
    plt.yscale("log")  # optional; helps when regrets span orders of magnitude
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_final_simple_regret_box(df_final):
    plt.figure(figsize=(10,6))
    sns.boxplot(data=df_final, x="method", y="simple_regret")
    plt.title("Final Simple Regret Distribution (across challenges & seeds)")
    plt.xlabel("Method")
    plt.ylabel("Final simple regret")
    plt.yscale("log")  # optional
    plt.xticks(rotation=30, ha="right")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()

def plot_auc_bar(table_B):
    plt.figure(figsize=(10,5))
    tb = table_B.sort_values("mean_rank_auc_simple_regret")
    sns.barplot(data=tb, x="method", y="mean_rank_auc_simple_regret")
    plt.title("Overall (Lower is Better): Mean Rank by AUC Simple Regret")
    plt.xlabel("Method")
    plt.ylabel("Mean rank (AUC-SR)")
    plt.xticks(rotation=30, ha="right")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()
```

***

## 8) How to relate this back to your RMSE evaluation

In your report, present it as:

*   **RMSE (prequential)** answers:  
    “How accurate is the surrogate at predicting the next observation?”

*   **Regret** answers:  
    “How good is the optimiser (surrogate + acquisition) at finding high objective values quickly?”

This is exactly the kind of multi-angle comparison used in BO research, where surrogate choice and acquisition aim to minimise regret (simple or cumulative) rather than purely predictive error. [\[jmlr.org\]](https://jmlr.org/papers/volume26/22-0523/22-0523.pdf), [\[proceeding....mlr.press\]](https://proceedings.mlr.press/v77/nguyen17a/nguyen17a.pdf), [\[nature.com\]](https://www.nature.com/articles/s41524-021-00662-x.pdf)

***

## 9) What you should do next (practical steps)

1.  Decide your **budget** per challenge (e.g., +20 evaluations beyond current)

2.  Choose **f\*** approach (known vs empirical best)

3.  Implement 3–6 methods, for example:
    *   GP + EI
    *   GP + UCB
    *   RF + UCB-style
    *   BART + EI (BART is explicitly used as a BO surrogate in literature) [\[nature.com\]](https://www.nature.com/articles/s41524-021-00662-x.pdf)
    *   MLP + EI (optional)

4.  Run each method with **N seeds** (e.g., 10–30)

5.  Produce:
    *   Table A (per challenge)
    *   Table B (overall ranks)
    *   Learning curves + box plots

***

### One thing I **won’t** assume (but you should check)

Do your challenges provide the true optimum $$f^\*$$?  
If yes, use it. If not, use the empirical best across all methods.

If you want, paste a tiny description of one challenge’s domain (continuous bounds, dimensionality, noise/noise-free), and I’ll tailor:

*   which acquisitions make sense,
*   how to do BART uncertainty for BO,
*   and what tuning strategy is safe with only \~15–30 evaluations.
